In [2]:
from src import extract, to_sql

In [3]:
periodos = ["201701", "201901", "202101", "202301", "202501"]
extract.download_cnes_zips(periodos, reprocess=False)
to_sql.process_cnes_zip(periodos, reprocess=False)

BASE_DE_DADOS_CNES_201701.ZIP já existe. Pulando...
BASE_DE_DADOS_CNES_201901.ZIP já existe. Pulando...
BASE_DE_DADOS_CNES_202101.ZIP já existe. Pulando...
BASE_DE_DADOS_CNES_202301.ZIP já existe. Pulando...
BASE_DE_DADOS_CNES_202501.ZIP já existe. Pulando...


Processando períodos: 100%|██████████| 5/5 [00:00<00:00, 6725.95it/s]

[AVISO] Pulando base existente: /home/phprestes/Documents/IC/data/02_intermediate/sql_cnes_201701.duckdb
[AVISO] Pulando base existente: /home/phprestes/Documents/IC/data/02_intermediate/sql_cnes_201901.duckdb
[AVISO] Pulando base existente: /home/phprestes/Documents/IC/data/02_intermediate/sql_cnes_202101.duckdb
[AVISO] Pulando base existente: /home/phprestes/Documents/IC/data/02_intermediate/sql_cnes_202301.duckdb
[AVISO] Pulando base existente: /home/phprestes/Documents/IC/data/02_intermediate/sql_cnes_202501.duckdb
Processamento finalizado com tabelas separadas por competência.


In [4]:
import duckdb
from src.constant import INTERMEDIATE_FOLDER

con201701 = duckdb.connect(INTERMEDIATE_FOLDER / 'sql_cnes_201701.duckdb', read_only=True)
con202501 = duckdb.connect(INTERMEDIATE_FOLDER / 'sql_cnes_202501.duckdb', read_only=True)

tabelas = con201701.execute("SHOW TABLES").fetchall()
print("Tabelas encontradas:")
for t in tabelas:
    print(f"- {t[0]}")
len(tabelas)

Tabelas encontradas:
- rlAdmGerenciaCnes201701
- rlCooperativa201701
- rlEquipeAldeia201701
- rlEquipeNasfEsf201701
- rlEstabAtenPsico201701
- rlEstabAtendPrestConv201701
- rlEstabAvaliacao201701
- rlEstabCentralReg201701
- rlEstabColetaSelRejeito201701
- rlEstabComissaoOutro201701
- rlEstabComplementar201701
- rlEstabEndCompl201701
- rlEstabEqpEmbarcacao201701
- rlEstabEqpUnidApoio201701
- rlEstabEquipamento201701
- rlEstabEquipeMun201701
- rlEstabEquipeProf201701
- rlEstabInstFisiAssist201701
- rlEstabOrgParc201701
- rlEstabPoloAldeia201701
- rlEstabProfComissao201701
- rlEstabProgFundo201701
- rlEstabRegimeRes201701
- rlEstabRepresentante201701
- rlEstabSamu201701
- rlEstabServClass201701
- rlEstabServicoApoio201701
- rlEstabSipac201701
- rlEstabSubTipo201701
- rlEstabTeleCnes201701
- rlEstabUnidAcolhim201701
- rlMunAtenPsico201701
- rlMunRegimeRes201701
- rlMunUnidAcolhim201701
- rlNasfEsf201701
- tbArea201701
- tbBanco201701
- tbCargaHorariaSus201701
- tbDadosProfissionalSus201701

53

In [5]:
from src.constant import FACT_TABLES
import pandas as pd

def analisar_tabela(table_name, con):
    # 1. Verificar tabela
    tables = [t[0] for t in con.execute("SHOW TABLES").fetchall()]
    if table_name not in tables:
        print(f"[ERRO] Tabela '{table_name}' não encontrada.")
        return None, 0
    
    try:
        # 2. Schema e Tipos
        schema_df = con.execute(f"DESCRIBE {table_name}").df()
        columns = schema_df['column_name'].tolist()
        
        # 3. Query única para Estatísticas Básicas (Total, Nulos, Únicos)      
        aggs = ["COUNT(*) as total_rows"]
        for col in columns:
            aggs.append(f"""COUNT("{col}") as "count_{col}" """)
            aggs.append(f"""COUNT(DISTINCT "{col}") as "unique_{col}" """)

        query_stats = f"SELECT {', '.join(aggs)} FROM {table_name}"
        stats_result = con.execute(query_stats).df()
        
        total_rows = stats_result['total_rows'][0]
        if total_rows == 0:
            print("[ERRO] A tabela está vazia.")
            return None, 0
            
        # 4. Descobrir a Moda e a Frequência da Moda
        mode_stats = {}
        for col in columns:
            # Pega o valor mais comum e sua contagem (ignorando nulos como é padrão na moda)
            query_mode = f"""
                SELECT "{col}" as mode_val, COUNT(*) as freq 
                FROM {table_name} 
                WHERE "{col}" IS NOT NULL 
                GROUP BY 1 
                ORDER BY 2 DESC 
                LIMIT 1
            """
            mode_res = con.execute(query_mode).fetchall()
            
            if mode_res:
                mode_stats[col] = {'val': mode_res[0][0], 'freq': mode_res[0][1]}
            else: # Caso a coluna seja 100% nula
                mode_stats[col] = {'val': None, 'freq': 0}
        
        # 5. Montar Relatório Final
        report_list = []
        
        for col in columns:
            count_val = stats_result[f"count_{col}"][0]
            unique_val = stats_result[f"unique_{col}"][0]
            
            mode_val = mode_stats[col]['val']
            mode_freq = mode_stats[col]['freq']
            
            # Cálculos de Porcentagem
            null_pct = ((total_rows - count_val) / total_rows) * 100
            unique_pct = (unique_val / total_rows) * 100
            
            # Nova métrica: % da Moda sobre o total de linhas
            mode_pct = (mode_freq / total_rows) * 100
            
            report_list.append({
                'Coluna': col,
                '% Nulos': round(null_pct, 4),
                '% Valores Únicos': round(unique_pct, 4),
                'Moda (Valor Comum)': str(mode_val)[:30], # Corta string longa se necessário
                '% Moda': round(mode_pct, 4)
            })
            
        return pd.DataFrame(report_list), total_rows

    except Exception as e:
        print(f"[ERRO] {e}")
        import traceback
        traceback.print_exc()
        return None, 0

In [6]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.colheader_justify', 'center')

def print_analysis(df_res, total):
        if df_res is not None:
                print("\n" + "="*80)
                print(f" -> RELATÓRIO ESTRUTURAL: {table_name}")
                print(f" -> Total de Linhas: {total}")
                print("="*80)
                display(df_res)
        else:
                print("[ERRO] A tabela está vazia.")

In [7]:
table_name = FACT_TABLES[3]
print_analysis(*analisar_tabela(table_name + '201701', con201701))
print_analysis(*analisar_tabela(table_name + '202501', con202501))


 -> RELATÓRIO ESTRUTURAL: rlEstabAtendPrestConv
 -> Total de Linhas: 599097


,Coluna,% Nulos,% Valores Únicos,Moda (Valor Comum),% Moda
0,co_unidade,0.0000,55.0360,3113702178982,0.0038
1,co_atendimento_prestado,0.0000,0.0012,02,75.1837
2,co_convenio,0.0000,0.0012,02,43.8742
3,co_usuario,0.9813,2.1628,FCES,8.0733
4,to_chardt_atualizacaoddmmyyyy,0.0270,0.6360,07/02/2017,1.0539
5,to_chardt_atualizacao_origemddmmyyyy,100.0000,0.0000,None,0.0000



 -> RELATÓRIO ESTRUTURAL: rlEstabAtendPrestConv
 -> Total de Linhas: 993135


,Coluna,% Nulos,% Valores Únicos,Moda (Valor Comum),% Moda
0,co_unidade,0.0000,56.3959,3113702178982,0.0023
1,co_atendimento_prestado,0.0000,0.0007,02,78.1855
2,co_convenio,0.0000,0.0007,02,47.2159
3,co_usuario,0.4670,2.0383,FCES,4.9739
4,to_chardt_atualizacaoddmmyyyy,0.0142,0.6124,19/07/2022,0.9534
5,to_chardt_atualizacao_origemddmmyyyy,100.0000,0.0000,None,0.0000


In [8]:
titulo = "# Relatório de Análise CNES\n\n"
descricao = "Este arquivo foi gerado automaticamente para documentar as estatística de cada coluna em cada uma das tabelas do CNES nas competências de 2017/01 e 2025/01.\n\n"
conteudo_final = titulo + descricao

# Removemos 'conteudo_final' dos parâmetros, a função agora apenas gera e retorna o texto
def to_markdown(table_name, comp, con):
    df, total = analisar_tabela(table_name + comp, con)
    
    # Dica: Adicionei a variável {comp} no título para você saber qual ano está lendo no relatório!
    texto_tabela = f"### RELATÓRIO ESTRUTURAL: {table_name} (Competência: {comp})\n"
    texto_tabela += f"**Total de Linhas:** {total}\n\n"
    
    if df is not None:
        texto_tabela += df.to_markdown(index=False) + "\n\n"
    else:
        texto_tabela += "> [ERRO] A tabela está vazia ou não foi encontrada.\n\n"
        
    return texto_tabela

# Agora concatenamos o resultado da função na variável principal
for table_name in FACT_TABLES:
    conteudo_final += to_markdown(table_name, '201701', con201701)
    conteudo_final += to_markdown(table_name, '202501', con202501)

# Salvando o arquivo
with open("relatorio_analise_dados.md", "w", encoding="utf-8") as f:
    f.write(conteudo_final)

print("Relatório gerado com sucesso!")

[ERRO] A tabela está vazia.
[ERRO] Tabela 'tbEstabBanco201701' não encontrada.
[ERRO] Tabela 'tbEstabBanco202501' não encontrada.
[ERRO] Tabela 'rlJustifPtProf201701' não encontrada.
[ERRO] Tabela 'rlJustifPtProf202501' não encontrada.
[ERRO] Tabela 'rlJustifPtProfLog201701' não encontrada.
[ERRO] Tabela 'rlJustifPtProfLog202501' não encontrada.
[ERRO] A tabela está vazia.
[ERRO] A tabela está vazia.
[ERRO] A tabela está vazia.
[ERRO] A tabela está vazia.
[ERRO] A tabela está vazia.
[ERRO] Tabela 'tbJustificaDesligaPrf201701' não encontrada.
[ERRO] Tabela 'tbJustificaDesligaPrf202501' não encontrada.
[ERRO] Tabela 'tbLocalGerenteAdministrador201701' não encontrada.
[ERRO] Tabela 'tbLocalGerenteAdministrador202501' não encontrada.
[ERRO] A tabela está vazia.
[ERRO] A tabela está vazia.
[ERRO] A tabela está vazia.
Relatório gerado com sucesso!
